# 08 - Training Diagnostics & Regularization

Module 02, notebook 03 taught you to read a learning curve for a scikit-learn model.
Neural networks need the same skill, plus a different toolkit for fixing what you find.

By the end you will be able to:
1. Read a train/validation loss curve and name the failure.
2. Diagnose a bad learning rate from the curve shape alone.
3. Apply dropout, weight decay and early stopping — and say what each one does.
4. Know which knob to reach for first.

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Deliberately hard: few samples, many features, plenty of noise. Overfitting is easy here.
X, y = make_classification(n_samples=600, n_features=40, n_informative=8,
                           n_redundant=5, flip_y=0.08, random_state=42)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_tr)

to_t = lambda a: torch.tensor(a, dtype=torch.float32)
X_train, X_test = to_t(scaler.transform(X_tr)), to_t(scaler.transform(X_te))
y_train, y_test = to_t(y_tr).reshape(-1, 1), to_t(y_te).reshape(-1, 1)

print("train", X_train.shape, "| validation", X_test.shape)

## 1. A reusable training function

Everything below changes one argument of this function and re-reads the curve.

In [ ]:
def build(n_features=40, hidden=(256, 256), dropout=0.0, seed=42):
    torch.manual_seed(seed)          # seed BEFORE building, so every model starts identical
    layers, prev = [], n_features
    for h in hidden:
        layers += [nn.Linear(prev, h), nn.ReLU()]
        if dropout > 0:
            layers.append(nn.Dropout(dropout))
        prev = h
    layers.append(nn.Linear(prev, 1))
    return nn.Sequential(*layers)

def train(model, epochs=300, lr=1e-3, weight_decay=0.0, patience=None):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    hist = {"train": [], "val": []}
    best_val, best_epoch, best_state, waited = float("inf"), 0, None, 0

    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()
        loss = criterion(model(X_train), y_train)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(X_test), y_test).item()

        hist["train"].append(loss.item())
        hist["val"].append(val_loss)

        if val_loss < best_val:
            best_val, best_epoch, waited = val_loss, epoch, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            waited += 1
            if patience is not None and waited >= patience:
                break

    hist["best_epoch"], hist["best_val"] = best_epoch, best_val
    hist["final_val"] = hist["val"][-1]
    hist["best_state"] = best_state
    return hist

def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        return ((torch.sigmoid(model(X)) > 0.5).float() == y).float().mean().item()

def plot(hist, title, ax):
    ax.plot(hist["train"], label="train", color="#4C78A8")
    ax.plot(hist["val"], label="validation", color="#F58518")
    ax.axvline(hist["best_epoch"], ls="--", c="grey", lw=1,
               label=f"best epoch {hist['best_epoch']}")
    ax.set_xlabel("epoch"); ax.set_ylabel("BCE loss")
    ax.set_title(title, fontsize=10); ax.legend(fontsize=8); ax.grid(alpha=.3)

## 2. What overfitting looks like

Train a large network with no regularization at all.

In [ ]:
baseline = build(hidden=(256, 256))
hist = train(baseline)

fig, ax = plt.subplots(figsize=(9, 4.5))
plot(hist, "No regularization", ax)
plt.show()

print(f"train accuracy      : {accuracy(baseline, X_train, y_train):.3f}")
print(f"validation accuracy : {accuracy(baseline, X_test, y_test):.3f}")
print(f"best validation loss was at epoch {hist['best_epoch']}, "
      f"but we trained for {len(hist['train'])}")

**The signature of overfitting in deep learning:** training loss keeps falling while
validation loss bottoms out and then turns *upward*. Everything after that turning point
is the network memorising noise.

Note the gap between the best epoch and the last epoch. Those wasted epochs actively made
the model worse — which is what the next four tools fix.

## 3. Learning rate — the first thing to check

Before reaching for regularization, check the learning rate. Its failures have distinctive
curve shapes.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, lr, label in zip(axes, [10.0, 1e-3, 1e-6],
                         ["Too high (10.0)", "About right (1e-3)", "Too low (1e-6)"]):
    h = train(build(hidden=(64, 64)), epochs=300, lr=lr)
    plot(h, label, ax)
plt.tight_layout()
plt.show()

| Curve shape | Diagnosis | Fix |
|---|---|---|
| Loss spikes, oscillates, or goes to `nan` | learning rate too **high** | divide by 10 |
| Loss falls smoothly then flattens with a rising validation line | about right, now overfitting | regularize |
| Loss barely moves, almost flat | learning rate too **low** | multiply by 10 |

Sweep by powers of ten (`1e-1, 1e-2, 1e-3, 1e-4`) — never in small increments.

## 4. Dropout

During training, dropout randomly zeroes a fraction of activations on every forward pass.
The network cannot rely on any single unit, so it is forced to spread the representation
out. At evaluation time it is switched off automatically by `model.eval()`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
results = []
for ax, p in zip(axes, [0.0, 0.3, 0.6]):
    m = build(hidden=(256, 256), dropout=p)
    h = train(m)
    plot(h, f"dropout = {p}", ax)
    results.append({"dropout": p,
                    "best val loss": round(h["best_val"], 3),
                    "final val loss": round(h["final_val"], 3),
                    "val acc": round(accuracy(m, X_test, y_test), 3)})
plt.tight_layout()
plt.show()

import pandas as pd
pd.DataFrame(results)

**Read the two loss columns separately — they tell different stories.**

- **`best val loss` barely moves.** Dropout did not find a better model than no dropout.
- **`final val loss` at 0.6 is clearly lower.** Dropout slowed the runaway; the curve
  climbs less steeply after the minimum.

That is the honest result on this dataset, and it is more typical than tutorials suggest.
Dropout is a brake on overfitting, not an accuracy booster. If you are already stopping at
the best epoch, the brake matters much less.

Watch the *gap* between the two lines in the plots — that is what dropout is compressing.
And note that too much dropout eventually hurts the training loss too, which is
under-fitting. **0.2 to 0.5 is the normal range**; 0.6 is aggressive.

## 5. Weight decay (L2)

This is `Ridge` from module 02, notebook 06, in a neural network. It penalises large
weights, so the model prefers a smoother function. In PyTorch it is one argument on the
optimizer.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
results = []
for ax, wd in zip(axes, [0.0, 1e-3, 1e-1]):
    m = build(hidden=(256, 256))
    h = train(m, weight_decay=wd)
    plot(h, f"weight_decay = {wd}", ax)
    results.append({"weight_decay": wd,
                    "val acc": round(accuracy(m, X_test, y_test), 3),
                    "best val loss": round(h["best_val"], 4)})
plt.tight_layout()
plt.show()

pd.DataFrame(results)

A small, consistent gain at `1e-3`; clear damage at `1e-1`. That is the usual shape —
useful values live in a narrow band around `1e-4` to `1e-3`, and the improvement is real
but small. Sweep it on a log scale, never linearly.

## 6. Early stopping — the one that actually matters here

The cheapest regularizer of all: stop when validation loss stops improving, and restore
the weights from the best epoch. It costs nothing and needs no tuning beyond `patience`.

In [ ]:
m = build(hidden=(256, 256))
h = train(m, epochs=300, patience=25)

print(f"stopped at epoch {len(h['train'])} of a possible 300")
print(f"best epoch was {h['best_epoch']}")

# Restore the best weights - without this, early stopping keeps the *worse* final weights.
m.load_state_dict(h["best_state"])
print(f"\nvalidation accuracy at best epoch: {accuracy(m, X_test, y_test):.3f}")

fig, ax = plt.subplots(figsize=(9, 4.5))
plot(h, "Early stopping with patience=25", ax)
plt.show()

**Compare the numbers.** The unregularized baseline ended around a validation loss of 1.4;
its best epoch was around 0.48. Early stopping alone recovers that entire difference — far
more than dropout and weight decay achieved between them.

That is the real lesson of this notebook. The glamorous knobs are worth a few percent.
Stopping at the right epoch is worth three times as much, and it is free.

> **The trap:** early stopping without restoring the best weights gives you the weights from
> `patience` epochs *after* the optimum — the ones that triggered the stop. Always keep and
> reload the best state, as above.

## 7. Everything together

In [ ]:
final = build(hidden=(256, 256), dropout=0.3)
h = train(final, epochs=400, lr=1e-3, weight_decay=1e-4, patience=40)
final.load_state_dict(h["best_state"])

fig, ax = plt.subplots(figsize=(9, 4.5))
plot(h, "dropout 0.3 + weight decay 1e-4 + early stopping", ax)
plt.show()

print(f"train accuracy      : {accuracy(final, X_train, y_train):.3f}")
print(f"validation accuracy : {accuracy(final, X_test, y_test):.3f}")
print(f"\nbaseline validation accuracy was {accuracy(baseline, X_test, y_test):.3f}")

## The order to reach for these

1. **Check the learning rate.** Free, and it is the problem more often than you expect.
2. **Add early stopping.** Free, no tuning, never hurts.
3. **Get more data, or augment it.** Beats every other item on this list.
4. **Add dropout** (0.2–0.5) or **weight decay** (1e-5 to 1e-3).
5. **Make the network smaller.** A model that cannot memorise the data cannot overfit it.
6. Only then reach for exotic architectures.

Beginners tend to start at 6. Start at 1.

### One honest caveat

On a dataset this small, the differences in `best val loss` are within noise from run to
run. Change the `seed=` argument of `build()` and that ranking may shuffle (the `final val
loss` ordering is more stable). That is not a flaw in the
notebook — it is the lesson from module 02, notebook 04, reappearing: **a single run is a
lottery ticket.** Before claiming a regularizer helped, run it with three seeds.

---
## Exercises

1. Run the dropout comparison with `build(..., seed=0)`, `seed=1`, `seed=2`. Does the
   ranking of 0.0 / 0.3 / 0.6 hold on `best val loss`? On `final val loss`? What does that
   tell you about single-run comparisons?
2. Set `hidden=(16, 16)` with no regularization. Does it still overfit? Relate the answer
   to item 5 in the list above.
3. Sweep `weight_decay` over `[0, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]` and plot best validation
   loss against it on a log x-axis. Where is the minimum?
4. Add `torch.optim.lr_scheduler.ReduceLROnPlateau` to the training loop, halving the
   learning rate when validation loss stalls. Does it beat a fixed rate?
5. Replace the full-batch loop with a `DataLoader` at `batch_size=32` (notebook 06,
   section 6). Does mini-batch noise act as a regularizer here?
6. **Stretch:** add `nn.BatchNorm1d` after each linear layer. Explain why batch norm
   behaves differently in `train()` and `eval()` mode, and what breaks if you forget.